<a href="https://colab.research.google.com/github/bsalami-092/Data_Science_Journey_Documentation/blob/main/Geospatial_Analysis_Assignment_1_with_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Data Acquisition
* Data Preprocessing
* Data Analysis
* Data Visualization

In [ ]:
# Import essential libraries
import pandas as pd # For data manipulation and analysis
import geopandas as gpd # For vector data handling
import rasterio # For raster data processing
import folium # For interactive maps
from shapely.geometry import Point  # For spatial operations
import ee # Google Earth Engine
import matplotlib.pyplot as plt # For visualization

In [ ]:

gdf = gpd.read_file('/content/grid3_nga_boundary_vaccstates.shp')

gdf.head()

In [ ]:
# Display the first five rows
gdf.head(5)

In [ ]:
# Check and fix CRS (Coordinate Reference System)
print(gdf.crs)

In [ ]:
# Convert to WGS 84 (EPSG: 4326) if needed
if gdf.crs != 'EPSG:4326':
  gdf = gdf.to_crs('EPSG:4326')

In [ ]:
# Visualize our shapefile
gdf.plot(figsize=(10, 6), column = 'statename', legend=True)
plt.title('States in Nigeria')
plt.show()

In [ ]:

ax = gdf.plot(figsize=(11, 8), column='statename', legend=True)
plt.title('States in Nigeria')

# Get the current legend (created by GeoPandas) and set its position
leg = ax.get_legend()
leg.set_bbox_to_anchor((1.05, 1))
leg.set_loc('upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Create a map centered at the average location
m = folium.Map(location=[gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()], zoom_start=10)

# Add markers and polygons
for _, row in gdf.iterrows():
  folium.Marker(
      location=[row.geometry.centroid.y, row.geometry.centroid.x],
      popup=row['capcity'] if 'capcity' in row else "No Name"
  ).add_to(m)

m # Display the map

In [ ]:
hf = '/content/GRID3_NGA_health_facilities_v2_0_5806009649412052847.csv'

In [ ]:
df =pd.read_csv(hf, encoding='latin-1')

df

In [ ]:
# Covert Latitude and Longitude to geometry
df['geometry'] = df.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)

# Convert to GeoDataFrame
g_df = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326') # WGS 84 CRS

g_df.head()

In [ ]:
# You can save the file into different format
gdf.to_file('health_facilities.shp')

In [ ]:
g_df.info()

In [ ]:
oyo_hf = df[df['state'] == 'Oyo']

oyo_hf.head()

In [ ]:
# Create map centered on the first point
m = folium.Map(location=[oyo_hf.iloc[0]['latitude'], oyo_hf.iloc[0]['longitude']], zoom_start=10)

# Add markers for each health facility
for _, row in oyo_hf.iterrows():
  folium.Marker(
      location=[row['latitude'], row['longitude']],
      popup=row['facility_name'],
      icon=folium.Icon(color='green')
  ).add_to(m)

# Display the map
m.save('map.html')
m

In [ ]:
oyo_hf.isna().sum()

In [ ]:
gdf_lga = gpd.read_file('/content/grid3_nga_boundary_vacclgas.shp')

gdf_lga.head()

In [ ]:
oyo_lga = gdf_lga[gdf_lga['statename'] == 'Oyo']

oyo_lga.head()

In [ ]:
# Clip the gdf_lga for Oyo State
clipped_gdf = gdf.clip(oyo_lga)

clipped_gdf.head()

In [ ]:
list(oyo_lga.lganame.unique())

In [ ]:
# Filter the data based on the selected lga
selected_lga = ['Ibarapa Central', 'Iseyin', 'Ibadan North' ]

filtered_lga = oyo_lga[oyo_lga['lganame'].isin(selected_lga)]

filtered_lga.head()

In [ ]:
# Filter the health data based on the selected lga
filtered_hf = oyo_hf[oyo_hf['lga'].isin(selected_lga)]

filtered_hf.head()

In [ ]:
# Initiate the map centered at Oyo
m = folium.Map(location=[filtered_hf.iloc[0]['latitude'], filtered_hf.iloc[0]['longitude']], zoom_start=10)

# Convert timestamp column to string before adding to GeoJson
filtered_lga.loc[:, 'timestamp'] = filtered_lga['timestamp'].astype(str)

# Add polygon layer (LGA Boundaries)
folium.GeoJson(filtered_lga, name='LGA Boundaries').add_to(m)

# Add points with popups
for idx, row in filtered_hf.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        popup=row['facility_name'],
        radius=5,
        color='blue',
        fill=True,
    ).add_to(m)

# Add Layer Control
folium.LayerControl().add_to(m)

m

In [ ]:
# To remove the outliers
filtered_hf = gpd.GeoDataFrame(filtered_hf, geometry='geometry', crs='EPSG:4326')
filtered_hf = gpd.sjoin(filtered_hf, filtered_lga, predicate='within')

filtered_hf.head()

In [ ]:
# Initiate the map centered at Oyo
m = folium.Map(location=[filtered_hf.iloc[0]['latitude'], filtered_hf.iloc[0]['longitude']], zoom_start=10)


# Add polygon layer (LGA Boundaries)
folium.GeoJson(filtered_lga, name='LGA Boundaries').add_to(m)

# Add points with popups
for idx, row in filtered_hf.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        popup=row['facility_name'],
        radius=5,
        color='blue',
        fill=True,
    ).add_to(m)

# Add Layer Control
folium.LayerControl().add_to(m)

m

In [ ]:
# Create a 500 meter buffer around points
filtered_hf_buffer = filtered_hf.copy()
filtered_hf_buffer['geometry'] = filtered_hf_buffer['geometry'].buffer(0.0045045045045)

filtered_hf_buffer

In [ ]:
# Initiate the map centered at Oyo
m = folium.Map(location=[filtered_hf.iloc[0]['latitude'], filtered_hf.iloc[0]['longitude']], zoom_start=10)

# Add polygon layer (LGA Boundaries)
folium.GeoJson(filtered_lga, name='LGA Boundaries').add_to(m)
folium.GeoJson(filtered_hf_buffer, name='Facility Buffer', style_function=lambda x:{
    'fillColor': 'yellow',
    'color': 'yellow',
    'weight': 1,
    'fillOpacity': 0.1
}).add_to(m)

# Add points with popups
for idx, row in filtered_hf.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        popup=row['facility_name'],
        radius=5,
        color='blue',
        fill=True,
    ).add_to(m)

# Add Layer Control
folium.LayerControl().add_to(m)

m